
---
title: "Bayesian Modelling for Industrial Applications: Hierarchical Regression"
date: 2025-12-10
description: "Learn how Bayesian regression works in practice using PyMC, through a real-world
  battery degradation case study. This article shows how probabilistic modeling
  improves prediction, uncertainty quantification, and decision-making in
  high-stakes industrial systems."

image: posterior_sensitivity.svg
twitter-card: 
    image: "posterior_sensitivity.svg"
open-graph: 
    image: "posterior_sensitivity.svg"

categories:
  - python
  - bayesian

title-block-banner: "bayesion-01.jpg"
format:
  html:
    code-fold: true
    code-summary: "Show the code"
    code-overflow: wrap
    shift-heading-level-by: 1
    reference-location: margin
    quarto-template-params:
      banner-header-class: "blog-post"
---

> **Series**: *Bayesian Modelling for Industrial Applications* – **Part 2**  
> **Prerequisite**: [Part 1 – Understanding Bayesian Thinking](https://sambaiga.github.io/blog/2025/10/bayesian-modelling-01.html)

## Introduction

Welcome back to our series on Bayesian Modelling for Industrial Applications. In [Part 2: Bayesian Regression](), we demonstrated how Bayesian regression provides a principled framework for continuous prediction under uncertainty. 
That discussion, however, treated all observations as if they arose from a single, homogeneous system. In real industrial operations, this assumption rarely holds. Industrial data is inherently structured, grouped, and imbalanced. 

Measurements are nested within components, components within subsystems, and fleets operate across diverse environments.  This post introduces **Hierarchical (Multilevel) Bayesian Regression**, the natural extension of Bayesian regression when data exhibits multi level structure. Hierarchical models allow us to learn individual behavior while still leveraging information from the entire population, striking a balance that is essential for modern industrial analytics.

## Motivation: Why Hierarchical Models?

[In Part 1](), we modeled battery capacity degradation using data from the [CALCE Battery Dataset](), focusing on how capacity evolves over cycle life. For clarity, we treated all observations as belonging to a single population. That assumption was intentional for introductory purposes but a closer inspection of the dataset reveals an important reality: not all batteries are the same.

The [CALCE dataset]() contains two battery chemistries, CX2 and CS2, each governed by distinct electrochemical processes and degradation mechanisms. Even within a single chemistry, no two cells are identical. Cells differ due to manufacturing batch effects, installation environments (e.g., thermal gradients based on rack position), and unique operating histories.

Compounding this heterogeneity is severe data imbalance. Some legacy cells have hundreds of capacity measurements, while newly deployed cells may have only two to five observations. Yet operational decisions such as maintenance scheduling, warranty validation, or derating must often be made early, precisely when data is scarcest.

This raises an unavoidable modeling question:

> How can we model battery degradation while accounting for chemistry-level differences and cell-level heterogeneity, without overfitting sparse data?

### The Pooling Problem in Battery Degradation

Treating all batteries as exchangeable members of a single population—our approach in Part 1 amounts to complete **pooling**. While this produces stable estimates, it obscures important structure:

1. Differences between CX2 and CS2 chemistries are averaged away
2. Cell-specific degradation patterns are lost
3. Predictions are biased toward an artificial “average battery”

At the opposite extreme, we could fit separate models for each cell or chemistry. This **no-pooling** approach captures heterogeneity, but it fails under realistic data conditions. Cells with only a handful of observations yield unstable estimates, exaggerated degradation rates, and unjustified confidence.

Industrial decisions are often required precisely when data is sparse. Neither extreme is acceptable.

## The Bayesian Resolution: Partial Pooling

Hierarchical Bayesian regression resolves this tension through **partial pooling**. Rather than forcing a choice between global and local models, hierarchical models assume that cell-specific parameters are related through higher-level population distributions.

In the battery context:

- Individual cells have their own degradation behavior
- Cells within the same chemistry share common characteristics
- Chemistries themselves may be related at a higher level

Information is shared across the dataset in a structured, probabilistic way. Data-rich cells are allowed to deviate from the population mean based on strong evidence, while data-sparse cells are gently shrunk toward chemistry-level behavior. This is not a heuristic trick—it is a direct consequence of Bayesian probability theory applied to structured data.

## A Hierarchical Degradation Model

Let $y_{ijk}$ denote the normalized capacity observed at cycle $c_{ijk}$ for observation $i$, cell $j$, belonging to chemistry $k$. Because capacity is bounded between 0 and 1, we model measurement noise using a Beta distribution:

$$
y_{ijk} \sim \mathrm{Beta}(\mu_{ijk}\phi, (1 - \mu_{ijk})\phi)
$$

where $\mu_{ijk}$ is the expected capacity and $\phi > 0$ is a global precision parameter that controls how tightly observations cluster around the mean. The mean is linked to a latent degradation state $\eta_{ijk}$ through a logit transform:

$$
\mathrm{logit}(\mu_{ijk}) = \eta_{ijk}.
$$

The latent state $\eta_{ijk}$ evolves according to a saturating exponential, reflecting the physical behavior of battery aging, which often slows after early rapid degradation:

$$
\eta_{ijk} = \alpha_{jk} - A_k \left(1 - e^{-\lambda_{jk} c_{ijk}} \right) + \mathbf{x}_{ijk}^\top \boldsymbol{\beta}.
$$

Here, $\alpha_{jk}$ is the initial logit-scale capacity for cell $j$, $\lambda_{jk}$ is its cell-specific degradation rate, and $A_k$ is the asymptotic degradation amplitude shared across all cells in chemistry $k$. Operational covariates $\mathbf{x}_{ijk}$ are multiplied by global effects $\boldsymbol{\beta}$, which are shared across all cells and chemistries. This structure ensures that early fast decay transitions naturally to slower long-term aging, consistent with electrochemical mechanisms.

---

### Global and Chemistry Levels

At the global level, hyperpriors govern the average behavior across all chemistries:

$$
\mu_{\alpha}^{\mathrm{global}} \sim \mathcal{N}(\alpha_0, 0.5^2), \quad
\mu_{\lambda}^{\mathrm{global}} \sim \mathcal{N}(\log(0.005), 0.5^2),
$$

$$
\sigma_{\alpha}^{\mathrm{global}} \sim \mathrm{HalfNormal}(0.5), \quad
\sigma_{\lambda}^{\mathrm{global}} \sim \mathrm{HalfNormal}(0.5), \quad
\sigma_A^{\mathrm{global}} \sim \mathrm{HalfNormal}(0.1).
$$

These hyperparameters determine how much chemistry-level parameters vary. Each chemistry $k$ draws its mean initial capacity and degradation rate from these global distributions:

$$
\mu_{\alpha,k} \sim \mathcal{N}(\mu_{\alpha}^{\mathrm{global}}, \sigma_{\alpha}^{\mathrm{global}}), \quad
\mu_{\lambda,k} \sim \mathcal{N}(\mu_{\lambda}^{\mathrm{global}}, \sigma_{\lambda}^{\mathrm{global}}),
$$

and the chemistry-level variability is modeled as

$$
\sigma_{\alpha,k} \sim \mathrm{HalfNormal}(0.2), \quad
\sigma_{\lambda,k} \sim \mathrm{HalfNormal}(0.2), \quad
A_k \sim \mathrm{HalfNormal}(\sigma_A^{\mathrm{global}}).
$$

This hierarchy allows chemistries to differ systematically while still borrowing strength from the global population.

---

### Cell Level and Shrinkage

Individual cells inherit their parameters from the chemistry distributions:

$$
\alpha_{jk} \sim \mathcal{N}(\mu_{\alpha,k}, \sigma_{\alpha,k}), \quad
\log \lambda_{jk} \sim \mathcal{N}(\mu_{\lambda,k}, \sigma_{\lambda,k}), \quad
\lambda_{jk} = \exp(\log \lambda_{jk}), \quad
A_{jk} = A_k.
$$

Shrinkage naturally arises from this structure. Cells with very little data are pulled toward their chemistry mean $\mu_{\alpha,k}$, and chemistry means themselves are pulled toward the global mean $\mu_{\alpha}^{\mathrm{global}}$. The strength of these pulls is determined by the variance parameters $\sigma_{\alpha,k}$ and $\sigma_{\alpha}^{\mathrm{global}}$. Small $\sigma_{\alpha,k}$ implies strong shrinkage toward the chemistry mean, while small $\sigma_{\alpha}^{\mathrm{global}}$ implies chemistries are pulled toward the global mean. Conversely, large variances allow the cell or chemistry parameters to deviate more from their parents. Mathematically, the posterior estimate of a cell's initial capacity can be approximated as

$$
\alpha_{jk} \approx \frac{\sigma_{\alpha,k}^{-2} \mu_{\alpha,k} + n_{jk} \hat{\alpha}_{jk}}{\sigma_{\alpha,k}^{-2} + n_{jk}},
$$

where $n_{jk}$ is the number of observations for that cell and $\hat{\alpha}_{jk}$ is the sample mean from the cell's data. Cells with sparse observations rely heavily on the chemistry mean, whereas cells with rich histories are informed primarily by their own data. This same logic applies at the chemistry level with respect to the global mean.

---

### Industrial Perspective

In practice, shrinkage ensures that early-life predictions are conservative: a newly deployed battery with only a few cycle measurements does not produce extreme degradation estimates, but instead leverages knowledge from its chemistry and the global population. As more observations accumulate, the model naturally trusts the individual cell trajectory, revealing true heterogeneity. This hierarchical structure turns sparse, imbalanced, and noisy datasets into a reliable decision-making system without relying on ad hoc regularization.



## Case Study: Battery Degradation with Hierarchical Structure


To make the discussion concrete, we return to the CALCE Battery Dataset, a widely used benchmark in battery health prognostics.  Rather than focusing on prediction alone, this case study uses the CALCE dataset to explore how degradation behavior is distributed across the system.

$$\text{Measurements} \rightarrow \text{Cells} \rightarrow  \text{Chemistry}$$

Two chemistries CX2 and CS2—form the top level. Within each chemistry, multiple individual cells are tested. Each cell contributes repeated capacity measurements over its cycle life. This structure allows us to move beyond a single degradation curve and instead ask where variability actually lives in the system. 

With a hierarchical Bayesian regression model in place, we can now investigate questions that were inaccessible in the single-population setting:

1. Chemistry-level behavior: Do CX2 and CS2 cells degrade at meaningfully different rates, or are observed differences largely due to cell-to-cell noise?

2. Cell-level variability: How much variation exists between individual cells within the same chemistry?

3. Variance decomposition: Is more uncertainty explained by chemistry differences or by individual cell behavior?

4. Learning dynamics: How quickly does uncertainty collapse as new measurements arrive for a previously unseen cell?

### Why This Matters for Decision-Making

One of the most practical benefits of hierarchical modeling is its ability to adapt confidence as evidence accumulates. For a newly deployed cell with only a few observations, predictions naturally reflect uncertainty informed by similar cells. As additional measurements arrive, the model updates smoothly, shifting weight from population-level behavior to cell-specific evidence.

This allows decisions to be made early—but not recklessly.

>Reflection Question:If cell-to-cell variability is very small compared to chemistry-level variability, what does this imply about manufacturing consistency? If the estimated cell-level variance collapses to nearly zero, how similar does the hierarchical model become to a complete pooling approach—and what does that tell us about the system?


In Part 1, Bayesian regression answered the question: "What is the relationship between capacity and cycle life?" Hierarchical Bayesian regression extends this by asking: "How does that relationship vary across real components in the system?". The single-level model from Part 1 becomes a special case when group-level variation is negligible.

In the sections that follow, we will formulate the hierarchical Bayesian regression model mathematically, showing how partial pooling emerges naturally from the model structure. 


In [ ]:
import arviz as az
from great_tables import GT, loc, md, style
from IPython.display import clear_output
from lets_plot import (
    LetsPlot,
    aes,
    coord_cartesian,
    facet_wrap,
    flavor_high_contrast_dark,
    geom_area,
    geom_band,
    geom_density,
    geom_histogram,
    geom_line,
    geom_point,
    geom_ribbon,
    geom_vline,
    gggrid,
    ggplot,
    ggsize,
    guide_legend,
    guides,
    labs,
    layer_tooltips,
    scale_color_brewer,
    scale_color_manual,
    scale_fill_manual,
    scale_y_continuous,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor as pt
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from bayes.plot.basic_plots import line_plot, modern_theme, pro_colors, scatter_plot
from bayes.plot.distribution import plot_density
from bayes.plot.style import configure_matlib_style

configure_matlib_style(style=["science", "arviz-doc", "tableau-colorblind10"], latex=True)

LetsPlot.setup_html(isolated_frame=False, offline=True, no_js=True, show_status=False)
np.random.seed(42)

### Level 2: The Cell-Level Model (Priors)
If we stopped here and just assigned independent priors to every $\alpha_j$ and $\lambda_j$, we would be doing "No Pooling" (overfitting small groups). Instead, we assume these cell-specific parameters come from a Chemistry-Level Normal Distribution.
For a cell $j$ belonging to chemistry $k$:Initial Capacity ($\alpha_j$):

$$\alpha_j \sim \mathcal{N}(\mu_{\alpha, k}, \sigma_{\alpha, k})$$

Degradation Rate ($\lambda_j$):Since rates must be positive, we model the log of the rate:
$$\log(\lambda_j) \sim \mathcal{N}(\mu_{\lambda, k}, \sigma_{\lambda, k})$$
Degradation Amplitude ($A_j$): 
$$A_j \sim \mathcal{N}(\mu_{A, k}, \sigma_{A, k})$$

This is the mathematical mechanism of Partial Pooling. The parameter $\sigma_{\lambda, k}$ represents the cell-to-cell variability within chemistry $k$. If $\sigma_{\lambda, k}$ is small, the model learns that all cells in that chemistry degrade very similarly, and it will aggressively "shrink" outliers toward the mean $\mu_{\lambda, k}$.
> Implementation Note: To avoid sampling issues (the "funnel" geometry), we implement this using the "offset" method. Instead of sampling $\alpha_j$ directly, we sample a standardized deviation $z_j \sim \mathcal{N}(0, 1)$ and transform it:

$$\alpha_j = \mu_{\alpha, k} + z_{\alpha, j} \cdot \sigma_{\alpha, k}$$

Applying the Matt Trick to our battery parameters, we define the following for a cell $j$ belonging to chemistry $k$:
1. Initial Capacity ($\alpha_j$):$$\alpha_j = \mu_{\alpha, k} + z_{\alpha, j} \cdot \sigma_{\alpha, k} \quad \text{where} \quad z_{\alpha, j} \sim \mathcal{N}(0, 1)$$2. Degradation Rate ($\lambda_j$):We model the rate on a log-scale to ensure it remains positive, then exponentiate:$$\lambda_j = \exp(\mu_{\lambda, k} + z_{\lambda, j} \cdot \sigma_{\lambda, k}) \quad \text{where} \quad z_{\lambda, j} \sim \mathcal{N}(0, 1)$$3. Degradation Amplitude ($A_j$):$$A_j = |\mu_{A, k} + z_{A, j} \cdot \sigma_{A, k}| \quad \text{where} \quad z_{A, j} \sim \mathcal{N}(0, 1)$$


>💡 Key Takeaway: The Matt Trick doesn't change the math of the model, but it changes the geometry of the problem, allowing the computer to find the solution much more reliably.

### Level 3: The Chemistry-Level Model (Hyper-priors)
Finally, we place "Hyper-priors" on the chemistry-level means and standard deviations. These distributions represent our prior belief about the battery technology before we see any specific cell data. For each chemistry 

- $k$:$\mu_{\alpha, k} \sim \mathcal{N}(2.0, 1.0)$ (Prior belief about average initial capacity)
- $\mu_{\lambda, k} \sim \mathcal{N}(\log(0.005), 0.5)$ (Prior belief about average decay rate
- $\sigma_{\dots, k} \sim \text{HalfNormal}(0.5)$ (Prior belief about manufacturing consistency)

These hyper-priors are "weakly informative"—they guide the model away from physically impossible values (like infinite capacity) but let the data speak for itself.

### Encoding Knowledge with Priors

In Bayesian modeling, defining priors is a critical step. This step allows domain knowledge accumulated from battery engineering to be embedded directly into the model, ensuring that predictions remain physically plausible even when data is sparse. A prior distribution is assigned to every unknown parameter ($\beta_0, \boldsymbol{\beta}, \lambda, \phi$). These priors act as soft constraints, preventing the model from learning extreme or non-physical relationships.

**The Intercept ($\beta_0$)** 

The Intercept $\beta_0$ represents the initial capacity of the battery fleet on the logit scale. The orange curve in the figure below represents the selected informative prior, $\text{Normal}(\mu_{\text{logit\_start}}, 0.5^2)$. A standard deviation of $\sigma = 0.5$ is chosen to balance prior knowledge (centering at $\mu_{\text{logit\_start}}$) with sufficient uncertainty to allow the observed data to meaningfully influence the final estimate.

```python
with pm.Model() as battery_model:
    eps=1e-8
    initial_logit_capacity_mean = -np.log(1-eps)
    intercept = pm.Normal("intercept", mu=initial_logit_capacity_mean, sigma=0.5)
```

In [ ]:
n = 1000
s1 = pm.draw(pm.Normal.dist(mu=0.28, sigma=0.1), n)
s2 = pm.draw(pm.Normal.dist(mu=0.28, sigma=0.2), n)
s3 = pm.draw(pm.Normal.dist(mu=0.28, sigma=0.5), n)

df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["(μ=0.28, σ=0.1)", "(μ=0.28, σ=0.2)", "(μ=0.28, σ=0.5)"], n),
    }
)

plot_density(df, title="Normal Distributions Intercept Priors", fig_size=(500, 400))

The narrower blue ($\sigma = 0.1$) and green ($\sigma = 0.2$) curves represent highly concentrated priors that would strongly restrict the posterior estimates. The wider $\sigma = 0.5$ (orange) distribution corresponds to a more conservative informative prior, granting the initial capacity estimate $\beta_0$ a reasonable degree of uncertainty.

**Operational Effects ($\boldsymbol{\beta}$)**

The vector of coefficients $\boldsymbol{\beta}$ controls the influence of operational features on capacity fade. Engineering knowledge suggests that, unless a feature is extreme, its immediate effect on capacity should be subtle, as the overall degradation process is primarily driven by cycle count.

```python
    with pm.Model() as battery_model:
    beta = pm.Normal("beta", mu=0, sigma=0.2, shape=n_features)
``` 


In [ ]:
from bayes.plot.distribution import plot_density

In [ ]:
n = 1000
s1 = pm.draw(pm.Normal.dist(mu=0, sigma=0.1), n)
s2 = pm.draw(pm.Normal.dist(mu=0, sigma=0.2), n)
s3 = pm.draw(pm.Normal.dist(mu=0, sigma=1.0), n)

df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["(μ=0, σ=0.1)", "(μ=0, σ=0.2)", "(μ=0, σ=1.0)"], n),
    }
)

plot_density(df, title="Normal Distributions Beta Priors", fig_size=(500, 400))

As shown in the figure above, a tight informative prior, $\text{Normal}(0, 0.2^2)$, is used for $\boldsymbol{\beta}$. Centering this prior at zero reflects the assumption that, on average, operational features have no effect, while the small standard deviation ($0.2$) requires strong evidence from the data before attributing a large effect to any single feature. This constraint prevents non-physical, abrupt changes in capacity predictions. In contrast, a broader prior such as $\text{Normal}(0, 1.0^2)$ (orange curve) allows extreme effects that are considered non-physical.


**Degradation rate $\lambda$** 

The degradation rate $\lambda$ governs the exponential decay term $e^{-\lambda k_i}$. Since degradation must always occur and capacity cannot increase indefinitely, it is necessary to enforce $\lambda > 0$. Accordingly, a Log-Normal prior, $\text{LogNormal}(\ln(0.005), 0.5^2)$, is used for $\lambda$.

```python
with pm.Model() as battery_model:
    lambda_rate = pm.Lognormal("lambda_rate", mu=np.log(0.01), sigma=0.5)
```
> 🧠 **Self-Test**: Recall that we set the prior for the fade rate $\lambda$ as $\text{LogNormal}(\ln(0.01), 0.5^2)$ (where $\sigma = 0.5$). What practical problem would arise if an engineer, overly confident in their historical knowledge, reset the prior to $\text{LogNormal}(\ln(0.01), 0.1^2)$ (where $\sigma = 0.1$)?

In [ ]:
n = 1000
s1 = pm.draw(pm.LogNormal.dist(np.log(0.005), sigma=0.1), n)
s2 = pm.draw(pm.LogNormal.dist(np.log(0.005), sigma=0.5), n)
s3 = pm.draw(pm.LogNormal.dist(np.log(0.005), sigma=1.0), n)

df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["(μ=In(0.005), σ=0.1)", "(μ=In(0.005), σ=0.5)", "(μ=In(0.005), σ=1.0)"], n),
    }
)

plot_density(df, title="LogNormal Distributions Priors", fig_size=(500, 400))

This weakly informative prior centers the expected degradation rate around $\mathbf{0.5\%}$, while the spread $\sigma = 0.5$ (green/teal curve) is sufficiently wide to accommodate realistic fleet-level variability. At the same time, it remains substantially tighter than $\sigma = 1.0$ (orange curve), thereby avoiding non-physical probability mass assigned to unrealistically large degradation rates.

This distribution reflects a conservative estimate of uncertainty, allowing greater variation in degradation behavior than a tighter prior (e.g., $\sigma = 0.1$) would permit, while still preventing implausible rates.

**Degradation Amplitude ($A$)**

The parameter degr_amp ($A$) controls the overall amplitude of the degradation component. Since this amplitude must be non-negative, a Half-Normal distribution is used, which has support only on positive values. The scale parameter $\sigma$ determines the strength of regularization.


```python
   with pm.Model() as battery_model:
    degr_amp = pm.HalfNormal("degr_amp", sigma=0.1)
```
 

In [ ]:
n = 1000
s1 = pm.draw(pm.HalfNormal.dist(sigma=0.1), n)
s2 = pm.draw(pm.HalfNormal.dist(sigma=0.2), n)
s3 = pm.draw(pm.HalfNormal.dist(sigma=0.5), n)


df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["σ=0.1", "σ=0.2", "σ=0.5"], n),
    }
)
plot_density(df, title="Gamma Distributions Phi Priors", fig_size=(500, 400))

As shown in the figure above, $\text{HalfNormal}(\sigma = 0.1)$ strongly concentrates probability mass near zero, requiring substantial evidence before attributing a large degradation amplitude. In contrast, broader priors such as $\text{HalfNormal}(\sigma = 0.5)$ place non-negligible probability on large, non-subtle amplitudes (up to approximately $1.0$), increasing the risk of overfitting by allowing the model to explain noise through the amplitude term.

**Precision Parameter ($\phi$)**

The precision parameter $\phi$ controls the variance of the Beta likelihood and represents the expected level of noise in the $\text{SoH}$ measurements. Accordingly, a highly informative Gamma prior, $\text{Gamma}(100, 2)$, is assigned to $\phi$.
```python
with pm.Model() as battery_model:
    phi = pm.Gamma("phi", alpha=100, beta=2.0)
```
This prior is centered at $\mathbb{E}[\phi] = \alpha / \beta = 50$ with a relatively small standard deviation ($\sigma_{\phi} = 5.0$), indicating high confidence in this expectation. This choice encodes the belief that sensor noise is low ($\sigma_{\text{noise}} \approx 0.14$), reflecting the physical reality of precise laboratory-grade measurements.

In [ ]:
n = 1000
s1 = pm.draw(pm.Gamma.dist(alpha=10, beta=1), n)
s2 = pm.draw(pm.Gamma.dist(alpha=50, beta=5), n)
s3 = pm.draw(pm.Gamma.dist(alpha=100, beta=2), n)


df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["Gamma(α=10, β=1)", "Gamma(α=50, β=5)", "Gamma(α=100, β=2)"], n),
    }
)
plot_density(df, title="Gamma Distributions Phi Priors", fig_size=(500, 400))


From the figure above, it is evident that $\text{Gamma}(\alpha = 100, \beta = 2.0)$ (orange curve) provides a strong belief in high precision. In contrast, $\text{Gamma}(\alpha = 10, \beta = 1.0)$ yields a lower expected precision with greater spread, allowing excessive uncertainty and risking a flat, unphysical prior predictive distribution. Alternative Gamma priors with the same expected precision but larger variance similarly underestimate the precision of modern sensors.

The complete model now combines all these components:
    

## Feature Engineering: Translating Raw Data to Diagnostics

In the preceding sections, the output side of the Bayesian model was rigorously defined, including the Beta likelihood, the Logit link function, and physics-informed priors for the parameters ($\beta_0, \boldsymbol{\beta}, \lambda, \phi$). However, the quality of the resulting predictions depends critically on the quality of the input features ($\mathbf{x}$) that drive the degradation term ($\eta_i = \dots + \mathbf{x}_i^{\top} \boldsymbol{\beta}$).

Raw capacity measurement curves are noisy and variable. Therefore, before proceeding to Bayesian sampling, it is necessary to dedicate a structured process to translating real-world operational data into robust, physically meaningful diagnostic features.




### Feature selection

After extracting a broad set of diagnostic and statistical features, feature selection is required. Using all available features can lead to overfitting, increased model complexity, and multicollinearity, which compromises interpretability of the Bayesian coefficients ($\boldsymbol{\beta}$). The final four features selected for regression ($\mathbf{x}$) are:
- charge_current_auc
- charge_current_mean
- discharge_voltage_auc
- discharge_voltage_crest


### Load pre-processed CALCE dataset

In [ ]:
FIGSHARE_DOWNLOAD_URL = "https://ndownloader.figshare.com/files/59415941"
features = ["discharge_voltage_crest"]
target = "capacity"
data = pd.read_parquet(
    FIGSHARE_DOWNLOAD_URL,
    engine="pyarrow",
    columns=["cycle", "BatteryID", "CellType", "failure_status"] + [target] + features,
)

In [ ]:
data["BatteryID"] = (
    data["BatteryID"].str.replace("CALCE_", "", regex=False).str.replace("2_", "", regex=False).str.replace("_", "")
)

In [ ]:
cell_idx, cell_labels = pd.factorize(data["BatteryID"])
unique_cells = data[["BatteryID", "CellType"]].drop_duplicates().set_index("BatteryID").reset_index()

In [ ]:
plot_density(data, x_col="capacity", color_col="CellType", title="")

In [ ]:
df_batteries = data[["BatteryID", "CellType"]].drop_duplicates()
spilit_ratio = 0.55
df_train = (
    df_batteries.groupby("CellType", group_keys=False)
    .apply(lambda g: g.sample(frac=spilit_ratio, random_state=42), include_groups=False)
    .reset_index(drop=True)
)
df_test = df_batteries[~df_batteries["BatteryID"].isin(df_train["BatteryID"])]
train_mask = data["BatteryID"].isin(df_train["BatteryID"])
train_df = data[train_mask].copy()
test_df = data[~train_mask].copy()

In [ ]:
plot_density(train_df, x_col="capacity", color_col="CellType", title="")

In [ ]:
# upper_bound, lower_bound = data[target].max(), data[target].min()

In [ ]:
def compute_logit(p, eps=1e-6):
    p_clipped = max(eps, min(p, 1 - eps))
    return np.log(p_clipped) - np.log(1 - p_clipped)


eps = 1e-6

In [ ]:
def hierarchical_beta_regression_model(
    data: pd.DataFrame,
    features: list[str],
    group_col: str = "chemistry",
    unit_col: str = "cell_id",
    target: str = "capacity",
    scaler: StandardScaler | None = None,
    lower_bound: float = 0.2,
    upper_bound: float = 1.3,
    eps: float = 1e-8,
) -> tuple[pm.Model, StandardScaler]:
    """Hierarchical Beta regression model with global hyperpriors for battery data.

    Structure:
        Global Hyper-priors -> Chemistry (Priors) -> Cell (Unit Parameters) -> Observation (Likelihood)
    """
    if scaler is None:
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(data[features])
    else:
        x_scaled = scaler.transform(data[features])

    target_scaler = MinMaxScaler(feature_range=(eps, 1 - eps))
    y_scaled = target_scaler.fit_transform(data[[target]].values.astype(np.float64)).flatten()

    cell_idx, cell_labels = pd.factorize(data[unit_col])
    unique_cells = data[[unit_col, group_col]].drop_duplicates().set_index(unit_col)
    chem_idx_per_cell = pd.factorize(unique_cells.loc[cell_labels, group_col])[0]
    cycles = data["cycle"].values.astype(np.float64)

    coords = {
        "cell": cell_labels,
        "chemistry": np.unique(unique_cells[group_col]),
        "obs": np.arange(len(data)),
        "features": features,
    }

    with pm.Model(coords=coords) as model:
        cycle_data = pm.Data("cycle_data", cycles, dims="obs")
        x_data = pm.Data("x_data", x_scaled, dims=("obs", "features"))
        y_data = pm.Data("y_data", y_scaled, dims="obs")

        cell_idx_pt = pm.Data("cell_idx", cell_idx, dims="obs")
        chem_idx_pt = pm.Data("chem_idx", chem_idx_per_cell, dims="cell")

        # ========== GLOBAL LEVEL FOR DEGRADATION PARAMETERS ==========
        # initial_logit_capacity_mean = -np.log(1 - 1e-6) ≈ 1e-6 ≈ 0
        initial_capacity_logit = -np.log(1 - 1e-6)

        mu_alpha_global = pm.Normal("mu_alpha_global", mu=initial_capacity_logit, sigma=0.5)
        mu_lambda_global = pm.Normal("mu_lambda_global", mu=np.log(0.005), sigma=0.5)

        sigma_alpha_global = pm.HalfNormal("sigma_alpha_global", sigma=0.5)
        sigma_lambda_global = pm.HalfNormal("sigma_lambda_global", sigma=0.5)
        amp_global = pm.HalfNormal("amp_global", sigma=0.1)
        beta = pm.Normal("beta", mu=0, sigma=0.2, dims="features")

        phi = pm.Gamma("phi", alpha=100, beta=2.0)

        # ========== CHEMISTRY LEVEL (DEGRADATION PARAMETERS) ==========
        mu_chem_alpha = pm.Normal("mu_chem_alpha", mu=mu_alpha_global, sigma=sigma_alpha_global, dims="chemistry")
        mu_chem_lambda = pm.Normal("mu_chem_lambda", mu=mu_lambda_global, sigma=sigma_lambda_global, dims="chemistry")

        sigma_chem_alpha = pm.HalfNormal("sigma_chem_alpha", sigma=0.2, dims="chemistry")
        sigma_chem_lambda = pm.HalfNormal("sigma_chem_lambda", sigma=0.2, dims="chemistry")

        amp_chem = pm.HalfNormal("amp_chem", sigma=amp_global, dims="chemistry")

        # ========== CELL LEVEL ==========
        # Cell-specific parameters come from chemistry-specific distributions

        # alpha_cell = pm.Normal(
        #    "alpha_cell", mu=mu_chem_alpha[chem_idx_pt], sigma=sigma_chem_alpha[chem_idx_pt], dims="cell"
        # )
        alpha_offset = pm.Normal("alpha_offset", mu=0, sigma=1, dims="cell")
        alpha_cell = pm.Deterministic(
            "alpha_cell", mu_chem_alpha[chem_idx_pt] + alpha_offset * sigma_chem_alpha[chem_idx_pt], dims="cell"
        )

        # lambda_cell = pm.Lognormal(
        #    "lambda_cell", mu=mu_chem_lambda[chem_idx_pt], sigma=sigma_chem_lambda[chem_idx_pt], dims="cell"
        # )

        lambda_offset = pm.Normal("lambda_offset", mu=0, sigma=1, dims="cell")
        log_lambda_cell = mu_chem_lambda[chem_idx_pt] + lambda_offset * sigma_chem_lambda[chem_idx_pt]
        lambda_cell = pm.Deterministic("lambda_cell", pm.math.exp(log_lambda_cell), dims="cell")

        amp_cell = amp_chem[chem_idx_pt]

        # ========== OBSERVATION MODEL ==========
        # Map cell parameters to observations
        lambda_obs = lambda_cell[cell_idx_pt]
        alpha_obs = alpha_cell[cell_idx_pt]
        amp_obs = amp_cell[cell_idx_pt]

        # Physical degradation model (domain knowledge): α - A * (1 - exp(-λ * cycles))
        degradation_curve = alpha_obs - amp_obs * (1 - pm.math.exp(-lambda_obs * cycle_data))

        # Feature effects (global)
        linear_effect = pm.math.dot(x_data, beta)

        # Combine on logit scale
        logit_mu = degradation_curve + linear_effect
        mu_scaled = pm.math.invlogit(logit_mu)

        # Beta likelihood with global precision
        alpha_dist = mu_scaled * phi
        beta_dist = (1 - mu_scaled) * phi
        pm.Beta("y_obs", alpha=alpha_dist, beta=beta_dist, observed=y_data, dims="obs")

        # ========== DETERMINISTIC QUANTITIES ==========
        # Predicted capacity (in scaled units)
        pm.Deterministic("capacity_pred_scaled", mu_scaled, dims="obs")

        # Feature effects
        pm.Deterministic("feature_effects", beta, dims="features")
    return model, scaler, target_scaler

In [ ]:
model, scaler, target_scaler = hierarchical_beta_regression_model(
    data=train_df,
    features=features,
    group_col="CellType",
    unit_col="BatteryID",
    target="capacity",
)

### Model building: Generalizability Test
Before training, we carefully partition the data to test the model's ability to generalize beyond the specific unit it learned from. This setup simulates a crucial real-world scenario where a model trained on a few prototype units must make predictions for an entire new batch of batteries.The strategy is as follows:

1. Isolate Cell Type: We first filter the entire dataset (data) to focus only on a single type of chemistry, Cell Type "CS2". This ensures the training and testing sets share fundamental physical properties, making the test a fair assessment of individual variability, not chemistry differences.
2. Train on One Unit: We designate a single, representative unit, "CALCE_CS2_38", as our train_df. The model will learn its degradation rate ($\lambda$) and operational sensitivities ($\boldsymbol{\beta}$) entirely from this unit's history.
3. Validate on the Fleet: The remaining batteries of the same chemistry are assigned to the test_df.This setup is intentionally challenging. Our model's success will be measured by its ability to accurately predict the capacity fade of the batteries in test_df units it has never seen before—by relying solely on the general parameters learned from the single training unit. This is the ultimate test of the model's generalizability.

In [ ]:
plots = []
for _, df in test_df.groupby("CellType"):
    for feature_name in features:
        plot = scatter_plot(df, y_col=feature_name)
        plots.append(plot + labs(title=feature_name) + modern_theme(font_size=9))


gggrid(plots=plots, ncol=2) + ggsize(650, 300)

The figure below plots the four selected features against cycle number for a representative battery. These plots empirically validate the selection process, as all four features exhibit clear, monotonic changes with cycling and, critically, show a distinct shift or acceleration in slope as the battery enters the failure state ($\text{SoH} \le 80\%$, shown in green). This strong visual correlation provides high confidence that these inputs will effectively drive the degradation component of our Bayesian model.

## Model Implementation and Inference

With our input data now rigorously cleaned, aligned, scaled, and reduced to the four most informative features ($\mathbf{X}$), we are ready to implement and fit our Bayesian regression. As defined in  [Beta Likelihood and Priors]() subsection, our model structure is a Bayesian Beta Regression. It combines our engineering knowledge (Priors) with the observed data (Likelihood) to model the bounded $\text{SoH}$ (scaled capacity $\mu_i$) via the Logit Link function.

We pass the four selected features: charge_current_auc, charge_current_mean, delta_voltage_variance, and discharge_voltage_crest, as the operational inputs (our $\mathbf{X}$ matrix) into the model definition.

In [ ]:
model

### Prior Predictive Check: Sanity Testing Our Assumptions

Following the rigorous Bayesian workflow, we begin with a crucial Prior Predictive Check (PPC). The PPC involves simulating data from the model using only the Prior Distributions, before seeing any observed data. This acts as a vital sanity test for our embedded engineering knowledge.

In [ ]:
with model:
    prior_pred = pm.sample_prior_predictive(samples=1000)

clear_output()
fig, ax = plt.subplots(figsize=(3.8, 1.8))
az.plot_ppc(prior_pred, group="prior", ax=ax)
plt.xlabel("Capacity")
plt.ylabel("Density");

The figure below, compare the predicted prior distribution (green line) against the observed data (blue line). This plot is essential for validating that our model's structural assumptions align with physical reality

In [ ]:
y_prior_flat = prior_pred.prior["capacity_pred_scaled"].stack(sample=["chain", "draw"]).values.flatten()
y_prior_flat = target_scaler.inverse_transform(y_prior_flat[:, None]).flatten()
y_obs = train_df[target].values

n_points = min(5000, len(y_prior_flat))
rng = np.random.default_rng(42)
idx = rng.choice(len(y_prior_flat), size=n_points, replace=False)
df_prior = pd.DataFrame(
    {"SoH": np.concatenate([y_obs, y_prior_flat[idx]]), "type": ["observed"] * len(y_obs) + ["prior"] * len(idx)}
)

In [ ]:
plot_density(
    df_prior,
    x_col="SoH",
    color_col="type",
    x_label="Capacity",
    fig_size=(400, 350),
    title="Prior comparison",
    subtitle="Prior Predictive Check",
)

The Prior Predictive Check (PPC) confirms the model's structural integrity, aligning with our tight prior specifications:

1. High Confidence Start: The predicted capacity is concentrated in a tight, dominant peak around $\approx 1.0$. This is directly enforced by the highly informative precision prior $\phi \sim \text{Gamma}(100, 2.0)$ (mean $\phi=50$), which ensures the model begins its estimation with high certainty (low noise).
2. Realistic Degradation Envelope: The prior distribution is highly constrained, lacking extreme spread across the capacity range. This controlled shape results from setting the tight standard deviation ($\sigma=0.1$) on the degradation rate prior ($\lambda$), minimizing the probability of immediate, catastrophic failure and constraining the model to plausible degradation trajectories.
3. Multimodal Coverage: The prior successfully allocates mass across lower capacity states (e.g., $0.4$ and $0.7$), reflecting the influence of the remaining uncertainty in $\lambda$ and $\text{degr\_amp}$. This structurally acknowledges degradation and failure possibilities without over-predicting their frequency.
   
Conclusion: The final model structure is sound, respecting physical bounds and combining high data quality confidence with a constrained, realistic acknowledgment of the degradation process. The model is now robust and ready for MCMC sampling.

## Running Inference (MCMC Sampling)

With the model fully defined, our priors validated via the PPC, and the data features prepared, we move to the final computational step: approximating the posterior distribution. This is the heart of Bayesian inference: transforming the prior uncertainty into an informed posterior distribution using the evidence from the data.

In [ ]:
with model:
    idata = pm.sample(500, tune=500, target_accept=0.90, random_seed=42, chains=2, cores=2, mp_ctx="spawn")
clear_output()


As discussed in [Part 1](https://sambaiga.github.io/blog/2025/10/bayesian-modelling-01.html), the MCMC process uses the No-U-Turn Sampler (NUTS) to explore the parameter space. The primary arguments guide this process:

- tune=2000: Specifies 2000 initial samples that are used solely to adapt the sampler's step size and are then discarded. A high tuning value is crucial for complex, highly curved posteriors (like those involving Beta distributions) to ensure stable exploration.
- draws=2000: Specifies 2000 final samples kept from the chain. These collected samples form the final Posterior Distribution for every model parameter ($\lambda$, $\beta$, $\phi$).
- target_accept=0.99: Forces the sampler to take smaller, more cautious steps. This high acceptance rate is necessary to avoid divergences in challenging models, ensuring a high-quality, accurate representation of the posterior distribution, though it increases computation time.
  
The resulting idata object now contains thousands of samples for every single model parameter, representing our comprehensive, uncertainty-quantified solution. The next step is to ensure these samples are reliable

### Model Diagnostics: Has the MCMC Converged?

After running the computationally intensive MCMC sampler, the very first step is to check the convergence diagnostics. The reliability of our final posterior estimates (our final answers) depends entirely on whether the Markov Chains successfully explored the entire parameter space. As discussed in [Part 1](), we focus on two key metrics shown in the summary table:

In [ ]:
vars = [
    "beta",
    "mu_alpha_global",
    "mu_lambda_global",
    "sigma_alpha_global",
    "sigma_lambda_global",
    "amp_global",
    "phi",
    "mu_chem_alpha",
    "mu_chem_lambda",
    "amp_chem",
]
data_summary = az.summary(idata, var_names=vars, kind="diagnostics")[["ess_bulk", "ess_tail", "r_hat"]]
GT(data_summary.reset_index()).tab_header(title="", subtitle="Diagnostics Summary").cols_label(
    {
        "ess_bulk": "ESS Bulk",
        "ess_tail": "ESS Tail.",
        "r_hat": "R-hat",
    }
)

All parameters show an $\hat{R}$ of exactly $1.0$, confirming the multiple Markov chains mixed exceptionally well and agree on the true posterior location. This indicates that the sampler thoroughly explored the parameter space without getting stuck in local modes. The high ESS values (ranging 4,126 to 5,915) confirm we collected a sufficient number of non-autocorrelated samples for every parameter, yielding highly reliable estimates for means and credible intervals.

With convergence confirmed, the sampled data is now a reliable representation of the Posterior Distribution. The next stage is to analyze these distributions to quantify the degradation process and the impact of our operational features.


### Analyzing the Posterior Distribution
With MCMC convergence successfully confirmed, we analyze the resulting Posterior Distribution to quantify our findings. The figure below displays the marginal Posterior Distribution (density, left) and the raw MCMC Trace Plot (right) for our core model parameters.

In [ ]:
az.plot_trace(idata, var_names=vars, compact=True);

## Interpreting the Posterior: Certainty, Rate, and Risk

The power of the Bayesian approach lies in its ability to quantify uncertainty for every parameter, moving beyond simple point estimates. The posterior summaries below allow us to identify reliable degradation drivers and estimate the full range of possible fade rates, which is crucial for managing risk in a battery fleet.

The analyze_parameter function below acts as a post-processing utility dedicated to generating publication-ready summary tables from the output of the MCMC sampling.

In [ ]:
def analyze_parameter(
    idata,
    parameter: str,
    features: list[str] | None = None,
    hdi_prob: float = 0.95,
    title: str = "Parameter Summary",
    subtitle: str | None = None,
) -> GT:
    """Generates a formatted summary table for a single parameter using Great Tables.

    This function extracts posterior summary statistics from ArviZ InferenceData and
    returns a beautifully styled table suitable for reports, notebooks, or publications.

    Args:
        idata: ArviZ InferenceData object containing posterior samples.
        parameter: Name of the parameter to summarize (e.g., "beta", "alpha", "sigma").
        features: List of feature names to label rows. Required and used only when
            ``parameter == "beta"``. Length must match the number of coefficients.
        hdi_prob: Highest density interval probability (default: 0.95).
        title: Main title for the table.
        subtitle: Optional subtitle. If None and parameter is "beta", defaults to
            "Beta coefficient analysis".

    Returns:
        A Great Tables (GT) object ready for display or further customization.

    Raises:
        ValueError: If ``features`` is provided for non-beta parameters or has wrong length.

    Example:
        >>> gt = analyze_parameter(idata, "beta", features=X.columns.tolist())
        >>> gt  # displays nicely in Jupyter
    """
    if features is not None and parameter != "beta":
        raise ValueError("`features` should only be provided when parameter == 'beta'")

    # Get summary statistics from ArviZ
    summary_df = az.summary(
        idata,
        var_names=[parameter],
        hdi_prob=hdi_prob,
        kind="stats",
        fmt="wide",
    ).reset_index(names="feature")

    # Assign meaningful feature names for beta coefficients
    if parameter == "beta":
        if features is None:
            raise ValueError("`features` must be provided when analyzing 'beta' parameter")
        if len(features) != len(summary_df):
            raise ValueError(
                f"Length of features ({len(features)}) must equal number of beta coefficients ({len(summary_df)})"
            )
        summary_df["feature"] = features

    conditions = [
        summary_df["hdi_2.5%"] > 0,  # Entire interval is positive
        summary_df["hdi_97.5%"] < 0,  # Entire interval is negative
    ]
    choices = ["Positive", "Negative"]
    summary_df["certainty"] = np.select(conditions, choices, default="Uncertain")

    # Set default subtitle for beta coefficients
    if subtitle is None and parameter == "beta":
        subtitle = "Beta coefficient analysis"

    gt_table = (
        GT(summary_df)
        .tab_header(
            title=md(f"**{title}**"),
            subtitle=md(subtitle) if subtitle else None,
        )
        .fmt_number(
            columns=["mean", "sd", "hdi_2.5%", "hdi_97.5%"],
            decimals=3,
        )
        .data_color(
            columns=["certainty"],
            palette=["#E1DFDD", "#F18F01", "#F18F01"],
            domain=["Uncertain", "Negative", "Positive"],
        )
        .cols_label(
            feature=md("**Feature**"),
            mean=md("**Mean**"),
            sd=md("**SD**"),
            **{"hdi_2.5%": md("**HDI 2.5%**")},
            **{"hdi_97.5%": md("**HDI 97.5%**")},
        )
        .cols_align(align="center", columns=["mean", "sd", "hdi_2.5%", "hdi_97.5%", "Certainty"])
        .tab_options(
            table_font_size="14px",
            heading_title_font_size="20px",
            heading_subtitle_font_size="16px",
            row_group_font_weight="bold",
        )
    )

    return gt_table

### Identifying Reliable Degradation Drivers ($\boldsymbol{\beta}$ Coefficients)

The $\boldsymbol{\beta}$ coefficients relate our features (like current and voltage metrics) to $\text{SoH}$ via the logit link function ($\log(\frac{\mu}{1 - \mu})$). We determine a predictor’s credibility by checking if its $95\%$ Highest Density Interval (HDI) crosses zero.

In [ ]:
analyze_parameter(idata, "beta", features=features)

The resulting summary above confirms that only one feature is a statistically reliable degradation driver at the $95\%$ confidence level: the discharge_voltage_crest factor. The discharge_voltage_crest has a 95% High Density Interval (HDI) that is entirely negative (from $-0.595$ to $-0.315$). This confirms, with high certainty, that an increase in this factor accelerates capacity fade.

The HDI for the other three features (charge_current_auc, charge_current_mean, and delta_voltage_variance) crosses zero (e.g., for charge_current_auc, the HDI is $[-0.421, 0.128]$). This means the model cannot rule out the possibility that their true effect is zero or even slightly positive. They are not statistically significant drivers.

Quantitative Impact: Focusing on the strongest driver, the mean coefficient for the discharge_voltage_crest factor is $\beta = -0.457$. We interpret this on the odds scale:$$\text{Odds Ratio} = \exp(-0.457) \approx 0.6$$This means that a one-unit increase in the crest factor is associated with the odds of high battery capacity decreasing by approximately $40\%$ ($\approx 1 - 0.615$).

Because all other features are not statistically supported, we can confidently focus our maintenance protocols on monitoring and controlling the discharge_voltage_crest factor. 

### Quantifying the Degradation Rate ($\lambda_{\text{rate}}$)

The $\lambda_{\text{rate}}$ parameter governs the speed of capacity fade. By using a less restrictive Lognormal prior ($\sigma=1.0$), the data was able to precisely estimate the actual degradation rate and quantify the remaining uncertainty.

In [ ]:
analyze_parameter(idata, "lambda_rate", title="Lambda Rate Summary", subtitle="Degradation rate parameter")

The resulting parameter summary provides a highly reliable estimate. The model's best estimate for the fade rate is $\mathbf{0.003}$ per unit of cycle data. This value is lower than the initial expected rate (from the prior centered around $0.01$). The small Standard Deviation ($\text{SD}=0.001$) and the narrow $95\%$ HDI ($[0.001, 0.006]$) demonstrate low remaining uncertainty. The MCMC has successfully used the data to achieve a highly precise estimate of the degradation speed.

###  Model Precision ($\phi$)
The $\phi$ parameter measures the model's precision, or how tightly the observed data clusters around the model's predicted mean after all factors are accounted for.

In [ ]:
analyze_parameter(idata, "phi", title="Phi Summary", subtitle="Phi  parameter")

The resulting posterior for $\phi$ shows a high degree of certainty, with a narrow $95\%$ HDI. The mean value of $91.255$ is the model's best estimate for the precision of the underlying process. This high mean value indicates that the residual variance (unexplained noise) in $\text{SoH}$ is very low. In practical terms, it means our combined model structure (the exponential term, $\lambda$, and the four operational features, $\boldsymbol{\beta}$) does a highly effective job of explaining the overall variability observed in the battery fleet. The tight HDI confirms that we have estimated this high level of precision with high certainty. 

### Analyzing the Posterior Distribution
Following parameter interpretation, the final diagnostic step is the Posterior Predictive Check (PPC). This step is essential for verifying that the model, using the now-informed posterior parameters, can generate synthetic data that closely resembles the actual observations. If the distributions of the synthetic and observed data align, we have high confidence that our model has captured the underlying data generating process.

We use PyMC's ``sample_posterior_predictive`` function to draw new samples from the likelihood distribution, using the parameters stored in the converged MCMC chains (idata).

In [ ]:
with model:
    post_pred = pm.sample_posterior_predictive(idata, var_names=["y_obs"], random_seed=42)

y_post = post_pred.posterior_predictive["y_obs"].stack(sample=["chain", "draw"]).values
# Mean of the posterior predictive
post_mean = y_post.mean()
n_draws = 500
rng = np.random.default_rng(42)
draw_idx = rng.choice(y_post.shape[0], size=n_draws, replace=False)
y_post_subset = y_post[:, draw_idx].flatten()


df_posterior = pd.DataFrame(
    {
        "SoH": np.concatenate([y_obs, y_post_subset]),
        "type": ["observed"] * len(y_obs) + ["posterior"] * len(y_post_subset),
    }
)

In [ ]:
df = pd.concat([df_prior, df_posterior])
plot_density(
    df,
    x_col="SoH",
    color_col="type",
    x_label="Capacity",
    fig_size=(500, 400),
    title="Prior and Posterior Comparison",
    subtitle="",
)

The figure above plots the observed capacity data against the synthetic data generated by the model. The plot shows the Posterior Predictive Distribution (blue) and the Observed Distribution (red) overlap significantly, particularly in the main body of the distribution (around 0.7). This strong visual alignment confirms that our complex Beta Regression model, incorporating the exponential fade and the four operational features, is an excellent fit for the underlying battery capacity data.

## Predict capacity for a new battery

Our final objective is to move from parameter estimation to practical Prognosis. We achieve this by generating a complete Posterior Predictive Distribution (PPD) for the capacity of a new or future battery state. This process leverages the full set of uncertainty-quantified model parameters ($\lambda_{\text{rate}}, \boldsymbol{\beta}, \text{and } \phi$). This process transforms our uncertainty-quantified parameter estimates into practical Prognostic predictions. This distribution automatically accounts for two critical types of uncertainty inherent in any prediction:
1. Epistemic Uncertainty: Uncertainty in the parameter estimates (e.g., how wide the HDI was for $\lambda_{\text{rate}}$).
2. Aleatoric Uncertainty: Inherent noise in the measurement process ($\phi$).

The output is not a single point, but a range ($\text{HDI}$) that tells us exactly where the true capacity is likely to fall.

To validate the model’s generalization capability and demonstrate its utility for risk management, we select three batteries from the CALCE dataset that were specifically excluded during model training. For each battery, we extract the same four features used in the original regression.



In [ ]:
from bayes.regression.beta_degradation import get_posterior_predictions

In [ ]:
new_df = test_df[test_df["BatteryID"].isin(["CALCE_CS2_34", "CALCE_CS2_36", "CALCE_CS2_37", "CALCE_CS2_33"])].copy()

The prediction process is summarized in the ``get_posterior_predictions`` function involves several key steps to generate predictions on new, unseen data using the fitted Bayesian model.:

- **Transform Input Features**: First, the new input feature data (data[features]) is transformed using the exact same scaler object that was fitted during the model training phase. This is crucial for ensuring the new data is on the same scale as the training data.

    ```python
    x_new = scaler.transform(data[features])
    ```
    Then we extracts the cycle numbers, which are a direct predictor variable required by the exponential degradation component in the Bayesian model.
   

- **Create Dummy Target Array**: The target variable container (y_obs in the PyMC model) must match the size of the new input data. We supply a dummy array (filled with an arbitrary value like $0.5$) solely to satisfy this structural size requirement, as its values are ignored during prediction.
   
   ```python
    y_dummy_scaled = np.full(shape=(X_new_scaled.shape[0],), fill_value=0.5)
   ```

- **Set New Data into Model**: Once all data arrays are prepared, the new feature data (x_new), cycle data (cycle_new), and the dummy target data (y_dummy_scaled) are set into the PyMC model's data containers using pm.set_data. This effectively prepares the model structure to run predictions on the new inputs.
  
   ```python
    with battery_model:
        pm.set_data({
            "x_data": x_new,
            "cycle_data": cycle_new,
            "y_obs": y_dummy_scaled
        })
   ```
- **Generate Posterior Predictive Samples**: Finally, we use the ```pm.sample_posterior_predictive``` function to generate predictions for the new battery data. This function draws samples from the Posterior Predictive Distribution (PPD), accounting for both the uncertainty in the model parameters (from idata) and the expected observation noise (from the Beta likelihood).
   
   ```python
    with battery_model:
        post_pred = pm.sample_posterior_predictive(
            idata,
            var_names=["y_obs"],
            random_seed=42,
            predictions=True,
        )
   ```


In [ ]:
pred_df = get_posterior_predictions(idata, model, scaler, new_df, features, alpha=0.1)

We then visualize the predictions for each of the three test batteries using the ```plot_hdi_regression``` function. This function plots the mean predicted capacity curve along with the $95\%$ HDI, overlaid with the actual observed capacity data for comparison.

The plots below show the observed capacity data (small points) against the model's Posterior Predictive Distribution (shaded area).
The blue line represents the Posterior Predictive Mean (the model's best guess for the expected capacity at any given cycle), and the shaded area represents the $95\%$ HDI of the predicted capacity, effectively quantifying our uncertainty about the battery's future state.

In [ ]:
from bayes.plot.regres_plot import plot_hdi_regression

In [ ]:
plot_hdi_regression(
    pred_df,
    x_column="cycle",
    y_column="capacity",
    group_column="BatteryID",
    pred_column="pred_median",
    x_label="Cycle Number",
    y_label="Capacity (Ah)",
    title_prefix="Battery Capacity vs. Cycle with Posterior Predictions",
    subtitle="90% HDI accounts for  uncertainty.",
    alpha=0.1,
) + ggsize(600, 500)

Key Observations from the Plot
1. The blue mean line tracks the observed capacity data closely across the entire life of the battery, including the non-linear degradation near the end of life (EOL).
2. The vast majority (ideally $\approx 95\%$) of the observed capacity points should fall within the shaded prediction interval, demonstrating that the model's estimated uncertainty is reliable (high PICP).
3.  The HDI band appears relatively narrow during the stable phase of the battery.  For cells CS2_33, CS2_36, and CS2_37, the HDI band narrows or remains tight through the final, steep degradation phase, suggesting high model confidence and low noise even during non-linear behavior. However, for cell CS2_34, the HDI band visibly widens toward the final cycles, reflecting increased model uncertainty.


### Evaluating Predictive Performance

We will evaluate the model's performance on the test data using a comprehensive set of metrics:

1. Accuracy ($R^2$, MAE and RMSE): These metrics measure how close the point prediction is to the true capacity value. For instance high $R^2$ indicates the model's mean curve captures the overall trend (degradation slope) well.
2. Reliability (PICP and NMPI): Measures the quality and tightness of the predicted uncertainty range ($\text{HDI}$). PICP measures the percentage of the true capacity observations that fall within the predicted HDI. On the other hand, NMPI quantifies the average width of the HDI relative to the range of observed capacities. A low NMPI indicates a tight uncertainty range, which is desirable for precise risk management.


The following code calculates these metrics for each individual test battery:

In [ ]:
from bayes.metrics.interval import get_interval_metrics
from bayes.metrics.regression import regression_report

In [ ]:
metrics_list = []
for battery_id, df in pred_df.groupby("BatteryID"):
    reg_report = regression_report(df["capacity"], df["pred_median"])
    interval_report = get_interval_metrics(
        df["pred_median"].values,
        df["capacity"].values,
        df["hdi_low"].values,
        df["hdi_high"].values,
        alpha=0.1,
    )
    full_report = pd.concat([reg_report, interval_report], ignore_index=True)
    full_report["BatteryID"] = battery_id
    metrics_list.append(full_report)
metrics_df = pd.concat(metrics_list, ignore_index=True)

metrics_df = metrics_df.pivot_table(
    index=["BatteryID"],
    columns="Metric",
    values="Value",
).reset_index()

In [ ]:
def make_metrics_table(metrics_df, title="Model Evaluation Results"):
    """Generate a formatted GT table from cross-validation metrics."""
    # Sort to present best models first
    df = metrics_df.sort_values(["BatteryID", "MAE"]).reset_index(drop=True)

    # Build base table
    gt = (
        GT(df[["BatteryID", "MAE", "RMSE", "R2", "NMPI", "PICP"]])
        .tab_header(title=title, subtitle="Per-battery performance")
        .cols_label(BatteryID="Test Cell", MAE="MAE", RMSE="RMSE", R2=md("R<sup>2</sup>"))
        .fmt_number(columns=["MAE", "RMSE", "NMPI", "PICP"], decimals=3)
        .fmt_number(columns="R2", decimals=3)
        .tab_spanner(label="Error Metrics", columns=["MAE", "RMSE", "NMPI", "PICP"])
        .tab_style(
            style=style.text(weight="bold"),
            locations=loc.body(columns="Model"),
        )
        .tab_options(
            table_font_size="small",
            # row_strip_color="#fafafa"
        )
    )
    for col in ["MAE", "RMSE", "R2", "NMPI", "PICP"]:
        best_idx = df[col].idxmax() if col in ["R2", "PICP"] else df[col].idxmin()
        gt = gt.tab_style(style=style.fill(color="#00B294"), locations=loc.body(rows=best_idx, columns=col))

    return gt

In [ ]:
make_metrics_table(metrics_df, title="Model Evaluation Results")

The predictive performance of the Bayesian Beta Regression model is quantitatively evaluated on the held-out test batteries. The results confirm the visual success demonstrated in the preceding plots

1. Accuracy: TThe model achieves a near-perfect fit ($R^2$ between $0.940$ and $0.990$) with minimal absolute error ($\text{MAE}$ of $0.020$ to $0.030$ SoH units). This high accuracy is visually confirmed by the blue mean line tracking the observed data almost perfectly in the plots.
2. Sharpness ($\text{NMPI}$): The Normalized Mean Prediction Interval is consistently low ($\mathbf{0.180 – 0.200}$). This confirms that the prediction interval is significantly narrower than the full capacity range, quantifying the low uncertainty visually represented by the tight, thin bands around the mean curve.
3. Reliability ($\text{PICP}$): The model demonstrates perfect $100\%$ coverage ($\text{PICP}=1.000$) for all four tested cells. This verifies that the prediction intervals are well-calibrated and trustworthy, representing a conservative but highly reliable outcome.

## Conclusion

We have successfully built a powerful single-level Bayesian model that provides trustworthy uncertainty bounds. This model provides highly accurate point predictions and reliable $95\%$ credible intervals, validating the use of informed priors and the Beta regression structure for modeling capacity degradation.However, we modeled the degradation rate ($\lambda_{\text{rate}}$) and operational effects ($\boldsymbol{\beta}$) as fixed across the entire fleet. 

In reality, manufacturing variation (such as slight manufacturing defects or differences in batch chemistry) means that Battery A might degrade faster than Battery B under the same conditions. A 'one-size-fits-all' model, even a very accurate one like this, is inherently limited because it cannot learn the unique individual characteristics of each unit, leading to either under-predicted Risk  for batteries in a "bad batch or "Over-predicted Caution for batteries in a "good batch.

**Coming Next**: Addressing Heterogeneity with Hierarchical Models. In Part 3, we will tackle this fundamental limitation by introducing Hierarchical Bayesian Models (HBMs). We will learn how to build a model that simultaneously learns a robust "Global Trend" for the entire fleet while allowing each individual battery to have its own informed "Local Deviation" for parameters like its degradation rate ($\lambda_{\text{rate}}$) and its sensitivity to operational features ($\boldsymbol{\beta}$).  This approach will significantly improve risk management and EOL prediction for large, heterogeneous battery fleets.

## References

1. Zhang, H., Li, Y., Zheng, S. et al. [Battery lifetime prediction across diverse ageing conditions with inter-cell deep learning](https://www.nature.com/articles/s42256-024-00972-x#citeas). Nat Mach Intell 7, 270–277 (2025). https://doi.org/10.1038/s42256-024-00972-x
2. Ferrari, S. L. P., & Cribari-Neto, F. (2004). Beta regression for modelling rates and proportions. Journal of Applied Statistics, 31(7), 799–815.
3. Gelman, A., Carlin, J. B., Stern, H. S., Dunson, D. B., Vehtari, A., & Rubin, D. B. (2013). Bayesian Data Analysis (3rd ed.). CRC Press.
4. McElreath, R. (2020). Statistical Rethinking (2nd ed.). CRC Press.